# Figure S1 — experiment-level QC

Two panels:

1. **Cells per gene knockout across experiments** — histogram of the per-experiment
   cells-per-geneKO statistic, log-scaled x, with the number of experiments on the y axis.
2. **Pairwise ISS correlation distribution** — histogram of pairwise Pearson correlations
   between experiments on log2 mean-normalised ISS barcode read frequencies.

For panel 1, an "experiment" is a single imaged channel or assay, split into four groups:

| Group | n | What it covers |
|---|---|---|
| Live-cell markers | 42 | GFP reporters, live-cell dyes, ChromaLIVE |
| Fixed-cell markers | 13 | cell painting (7 channels) + 4i (6 channels) |
| Phase | 1 | label-free, imaged in every OPS experiment |
| CROP-Seq | 1 | scRNA-seq |

**Inputs** (placed under `data/figures/SI/` by `data_preprocessing/figure_S1.py`):
`cells_per_gene_by_experiment.csv` and `iss_barcode_freq_correlation_matrix.csv`.
**Outputs** are written to `output/SI/`.

Note that channels within the cell-painting panel share one cells-per-gene value, and
channels within the 4i panel share another, because each panel's channels are read off
the same fixed cells. The fixed-cell group therefore lands as two tall bars rather than a
spread. The same holds for two co-imaged live-cell pairs (5xUPRE/ER_SEC61B,
MKI67/peroxisome_Peroxi_SPY650).

## Parameters

In [ ]:
# Which per-gene statistic to histogram: "mean_cells_per_gene" or "median_cells_per_gene".
STAT = "mean_cells_per_gene"
AXIS_PAD = 1.5         # multiplicative padding on the x range beyond the data extremes

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, NullLocator

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42

FIGURE_DATA = Path("../../data/figures/SI")
OUT_DIR = Path("../../output/SI")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load

One row per experiment. The `modality` column from the preprocessing script is collapsed
into the four plotting groups — cell painting and 4i both fold into `Fixed-cell markers`.

In [ ]:
CSV_PATH = FIGURE_DATA / "cells_per_gene_by_experiment.csv"

# Group order is also the plotting / legend / color-slot order.
GROUPS = ["Live-cell markers", "Fixed-cell markers", "Phase", "CROP-Seq"]
MODALITY_TO_GROUP = {
    "live_cell":     "Live-cell markers",
    "cell_painting": "Fixed-cell markers",
    "4i":            "Fixed-cell markers",
    "phase":         "Phase",
    "cropseq":       "CROP-Seq",
}

df = pd.read_csv(CSV_PATH)
df["group"] = df["modality"].map(MODALITY_TO_GROUP)
assert df["group"].notna().all(), f"unmapped modalities: {sorted(df.loc[df['group'].isna(), 'modality'].unique())}"

print(f"{len(df)} experiments, {df['n_genes'].iloc[0]} gene KOs each")
df.head()

## Per-group summary

Number of experiments per group and the range of the plotted statistic within each.
Also written to `figure_S1_group_summary.csv` in the output dir.

In [ ]:
summary = (
    df.groupby("group")[STAT]
    .agg(n_experiments="size", min="min", median="median", max="max")
    .reindex(GROUPS)
    .round(1)
)

SUMMARY_CSV = OUT_DIR / "figure_S1_group_summary.csv"
summary.to_csv(SUMMARY_CSV)
print(f"wrote {SUMMARY_CSV}")

summary

## Histogram

Shared log-spaced bins across all four groups, stacked so bar height reads directly as a
count of experiments (the fixed-cell and live-cell ranges overlap, so stacking rather than
overlaying keeps the total honest). Group identity is carried by the legend; the table view
below is the fallback for the color encoding.

In [ ]:
# Categorical slots 1-4 of the validated default palette, in fixed order.
GROUP_COLORS = {
    "Live-cell markers":  "#2a78d6",
    "Fixed-cell markers": "#eb6834",
    "Phase":              "#1baf7a",
    "CROP-Seq":           "#eda100",
}

# Explicit log-axis ticks, labelled as plain integers rather than powers of ten.
XTICKS = [500, 1000, 2000, 3000, 5000, 10000, 20000, 30000, 50000]

lo, hi = df[STAT].min() / AXIS_PAD, df[STAT].max() * AXIS_PAD
n_bins =  40
bins = np.logspace(np.log10(lo), np.log10(hi), n_bins + 1)

series = [df.loc[df["group"] == g, STAT].to_numpy() for g in GROUPS]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.hist(
    series,
    bins=bins,
    stacked=True,
    color=[GROUP_COLORS[g] for g in GROUPS],
    edgecolor="black",
    linewidth=0.8,
    label=[f"{g} (n={len(s)})" for g, s in zip(GROUPS, series)],
)

ax.set_xscale("log")
ax.set_xlim(bins[0], bins[-1])
# set_xticks alone leaves the default decade minor ticks in place, so clear them.
ax.set_xticks(XTICKS, labels=[str(t) for t in XTICKS], fontsize=9)
ax.xaxis.set_minor_locator(NullLocator())
ax.set_xlabel(f"Cells per gene knockout ({STAT.split('_')[0]} across gene KOs)")
ax.set_ylabel("Number of experiments")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


ax.margins(y=0.18)
ax.legend(frameon=False, fontsize=9, loc="upper right")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_S1_cells_per_gene_histogram.svg", bbox_inches="tight")
fig.savefig(OUT_DIR / "figure_S1_cells_per_gene_histogram.png", dpi=200, bbox_inches="tight")
plt.show()

## Table view

Every experiment behind the histogram, sorted by the plotted statistic — the accessible
fallback for the color encoding above.

In [ ]:
table = (
    df[["experiment", "group", "modality", STAT]]
    .sort_values(STAT, ascending=False)
    .reset_index(drop=True)
)
table

## Pairwise ISS correlation distribution

Moved here from `notebooks/figure_1/iss_correlation_heatmap.ipynb`, which retains the
heatmap of the full matrix.

Histogram of all unique pairwise correlations (upper triangle, excluding self-correlation)
between OPS experiments, computed on log2 mean-normalised ISS barcode read frequencies — a
QC measure of how consistently the barcode library is recovered across experiments. The
mean correlation is marked.

In [ ]:
# Experiment x experiment Pearson correlation matrix, curated into data/figures/SI/
# by data_preprocessing/figure_S1.py. figure_1.py copies the same upstream file into
# data/figures/figure_1/ for the heatmap.
CORR_MATRIX_CSV = FIGURE_DATA / "iss_barcode_freq_correlation_matrix.csv"

corr_df = pd.read_csv(CORR_MATRIX_CSV, index_col=0)
corr_values = corr_df.values.astype(float)
print(f"Loaded {corr_df.shape[0]}x{corr_df.shape[1]} correlation matrix")

pairwise = corr_values[np.triu_indices_from(corr_values, k=1)]
pairwise = pairwise[~np.isnan(pairwise)]

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(pairwise, bins=50, edgecolor="black")
ax.axvline(
    pairwise.mean(),
    color="red",
    linestyle="--",
    lw=2,
    label=f"mean = {pairwise.mean():.3f}",
)
ax.set_xlabel("Pairwise Pearson correlation")
ax.set_ylabel("Number of experiment pairs")
ax.set_title(f"Pairwise correlation distribution — {len(pairwise)} pairs")
ax.legend()

fig.tight_layout()
fig.savefig(OUT_DIR / "figure_S1_iss_correlation_histogram.svg", bbox_inches="tight")
fig.savefig(OUT_DIR / "figure_S1_iss_correlation_histogram.png", dpi=200, bbox_inches="tight")
plt.show()